In [1]:
import cv2
import tensorflow.keras as keras
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Dense, Flatten, Conv2D, MaxPooling2D
import keras_tuner as kt
from keras import regularizers

I0000 00:00:1780506519.752514   15864 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780506526.141727   15864 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780506552.960981   15864 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Extração das Imagens

In [4]:
count = -1
vid = cv2.VideoCapture("/mnt/Data/Lab---IA/videos/video_completo.mp4")

success = True

while success:
    success, image = vid.read()

    if success:
        cv2.imwrite(
            f"/mnt/Data/Lab---IA/imagens/frame{count+1}.jpg",
            image
        )
        count += 1

vid.release()

## Aplicação do Haarcascade

vide: https://patotricks15.medium.com/detec%C3%A7%C3%A3o-de-faces-com-python-opencv-modelo-haar-cascade-485bc1bcb368

In [ ]:
def haarcascade(num_fotos, nome_pessoa):
    for i in range(num_fotos):
        face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

        img = cv2.imread(f"/mnt/Data/LAB-I/imagens_cruas/{nome_pessoa}/{nome_pessoa}{i}.jpg")
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(gray, 1.3, 5)

        for (x,y,w,h) in faces:
            face = gray[y:y+h, x:x+w]
            face32x32 = cv2.resize(face, (32,32))
            cv2.imwrite(f"/mnt/Data/LAB-I/imagens_tratadas/{nome_pessoa}/{nome_pessoa}{i}.jpg", face32x32)

haarcascade(1469, "diego")
haarcascade(1579,"gabriel")
haarcascade(1931, "horacio")
haarcascade(1822, "jose")
haarcascade(1191, "julia")
haarcascade(2690, "lucio")
haarcascade(1586, "bruno")
haarcascade(1469, "samuel")
haarcascade(495, "rafael")
haarcascade(1741, "yuri")

O Haarcascade detecta algumas coisas que não são rostos. É preciso algum tratamento seguinte para adequar as minhas imagens. Salvei as imagens já filtradas (só rostos) em "/mnt/Data/LAB-I/haarcascade/". Não fiz isso com o dataset das pessoas aleatórias.

## Criando o Dataset

In [ ]:
# Pegando o dataset de outros rostos

import kagglehub

path = kagglehub.dataset_download("ashwingupta3012/human-faces")

print("Path to rotulos files:", path)

In [ ]:
num_fotos = 3000

for i in range(num_fotos):
    face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

    img = cv2.imread(f"/home/gwolfovitch/.cache/kagglehub/datasets/ashwingupta3012/human-faces/versions/1/Humans/1 ({i+1}).jpg")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x,y,w,h) in faces:
        face = gray[y:y+h, x:x+w]
        face32x32 = cv2.resize(face, (32,32))
        cv2.imwrite(f"/mnt/Data/LAB-I/imagens_tratadas/outros/face{i+1}.jpg", face32x32)

In [ ]:
def criar_dataset_autorizado(num_haarcascade, nome, pos_rotulo):
    dataset = np.zeros((num_haarcascade, 32, 32))
    rotulos = np.zeros((num_haarcascade, 11))
    for i in range(num_haarcascade):
        imagem = cv2.imread(f"/mnt/Data/LAB-I/imagens_tratadas/{nome}/{nome}{i}.jpg", cv2.IMREAD_GRAYSCALE)
        dataset[i] = imagem
        rotulos[i][pos_rotulo] = 1
    return dataset, rotulos

dataset_gabriel, rotulos_gabriel = criar_dataset_autorizado(1453, "gabriel", 0)
dataset_horacio, rotulos_horacio = criar_dataset_autorizado(1187, "horacio", 1)
dataset_lucio, rotulos_lucio = criar_dataset_autorizado(1320, "lucio", 2)
dataset_yuri, rotulos_yuri = criar_dataset_autorizado(1007, "yuri", 3)
dataset_samuel, rotulos_samuel = criar_dataset_autorizado(814, "samuel", 4)
dataset_julia, rotulos_julia = criar_dataset_autorizado(525, "julia", 5)
dataset_diego, rotulos_diego = criar_dataset_autorizado(261, "diego", 6)
dataset_jose, rotulos_jose = criar_dataset_autorizado(1206, "jose", 7)
dataset_rafael, rotulos_rafael = criar_dataset_autorizado(138, "rafael", 8)
dataset_bruno, rotulos_bruno = criar_dataset_autorizado(1052, "bruno", 9)

datasetOutros = np.zeros((2053, 32, 32))
rotulosOutros = np.zeros((2053, 11))

for i in range(2053):
    imagemOutros = cv2.imread(f"/mnt/Data/LAB-I/imagens_tratadas/outros/outros{i}.jpg", cv2.IMREAD_GRAYSCALE)
    datasetOutros[i] = imagemOutros
    rotulosOutros[i][10] = 1

datasetTotal = np.concatenate((dataset_gabriel, dataset_bruno, dataset_diego, dataset_horacio, dataset_lucio, dataset_yuri, dataset_samuel, dataset_julia, dataset_jose, dataset_rafael, datasetOutros))
rotulosTotal = np.concatenate((rotulos_gabriel, rotulos_bruno, rotulos_diego, rotulos_horacio, rotulos_lucio, rotulos_yuri, rotulos_samuel, rotulos_julia, rotulos_jose, rotulos_rafael, rotulosOutros))

datasetTotal = datasetTotal[:, :, :, np.newaxis] / 255

## Criação da Rede e Treino da Rede

In [ ]:
def construtor_modelo(hp):
    modelo = Sequential()

    modelo.add(InputLayer(shape =(32,32,1)))

    hp_regularizer = hp.Choice("hp_reg", values=[1e-2, 1e-3, 1e-4])
    
    modelo.add(Conv2D(
        filters=4, 
        kernel_size=(3,3), 
        kernel_regularizer=regularizers.L2(hp_regularizer), 
        activation="relu", 
        use_bias=True))

    modelo.add(MaxPooling2D(pool_size=(2, 2)))

    modelo.add(Flatten())

    hp_units = hp.Int('units', min_value=32, max_value=512, step=32)
    modelo.add(Dense(
        units=hp_units, 
        activation="relu", 
        use_bias=True))
    
    modelo.add(Dense(
        units=hp_units, 
        kernel_regularizer=regularizers.L2(hp_regularizer), 
        activation="relu", 
        use_bias=True))

    modelo.add(Dense(
        units=11, 
        activation="softmax", 
        use_bias=True))

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
    loss="categorical_crossentropy",
    metrics=['accuracy']
    )

    return modelo

tuner = kt.Hyperband(construtor_modelo,
                     objective='val_accuracy',
                     max_epochs=20,
                     factor=3,
                     directory='/mnt/Data/LAB-I/modelo/',
                     project_name='keras_tuner')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(datasetTotal, rotulosTotal, test_size=0.2, random_state=42)

tuner.search(X_train, y_train, epochs=50, validation_split=0.2)

best_hps=tuner.get_best_hyperparameters(num_trials=1)[0]

In [ ]:
model = tuner.hypermodel.build(best_hps)
history = model.fit(X_train, y_train, epochs=50, validation_split=0.2)

val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1

hypermodelo = tuner.hypermodel.build(best_hps)

hypermodelo.fit(X_train, y_train, epochs=best_epoch, validation_split=0.2)

In [ ]:
hypermodelo = tuner.hypermodel.build(best_hps)

In [ ]:
eval_result = hypermodelo.evaluate(X_test, y_test)

print("[test loss, test accuracy]:", eval_result)

hypermodelo.summary()

In [ ]:
path = "/mnt/Data/LAB-I/modelo/modelo_final/classificador_multiclasse.keras"
hypermodelo.save(path)

## Inferência

In [8]:
def float_to_q1_7(value):
    # 1. Scaling for 7 fractional bits
    scaled = round(value * (2**7))
    
    # 2. Saturation (Clamping for signed 8-bit: -128 to 127)
    # This prevents overflow/underflow from wrapping around
    if scaled > 127: scaled = 127
    if scaled < -128: scaled = -128
    
    # 3. Two's Complement Conversion
    # If negative, we mask it to 8 bits to get the positive integer representation
    if scaled < 0:
        binary_repr = (1 << 8) + scaled
    else:
        binary_repr = scaled
        
    return binary_repr, scaled

# Example usage:
val = -0.5
hex_val, integer_val = float_to_q1_7(val)

"""print(f"Float: {val}")
print(f"Integer (Fixed): {integer_val}")
print(f"Binary: {bin(hex_val)}")
print(f"Hex for FPGA: 8'h{hex_val:02X}")"""

'print(f"Float: {val}")\nprint(f"Integer (Fixed): {integer_val}")\nprint(f"Binary: {bin(hex_val)}")\nprint(f"Hex for FPGA: 8\'h{hex_val:02X}")'

In [ ]:
# Tratamento da imagem, passando pelo Haarcascade e transformando em uma matriz 32 x 32

face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

img = cv2.imread(f"/run/media/gwolfovitch/Data/LAB-I/images_teste/inf2.jpg")

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

faces = face_cascade.detectMultiScale(gray, 1.3, 5)

for (x,y,w,h) in faces:
    face = gray[y:y+h, x:x+w]
    face32x32 = cv2.resize(face, (32,32))
    img_infer = face32x32 / 255

img_infer = img_infer[np.newaxis, :, :, np.newaxis]

np.save("output32x32.npy", face32x32)
print(face32x32)

In [6]:

matrix_32x32 = cv2.imread(f"/run/media/gwolfovitch/Data/LAB-I/imagens_tratadas/imagens_teste_verilog/1.jpg")
gray = cv2.cvtColor(matrix_32x32, cv2.COLOR_BGR2GRAY)
face32x32 = cv2.resize(gray, (32,32))
print(face32x32.shape)

(32, 32)


In [13]:
for i in range(14):
    matrix_32x32 = cv2.imread(f"/run/media/gwolfovitch/Data/LAB-I/imagens_tratadas/imagens_teste_verilog/{i+1}.jpg")
    gray = cv2.cvtColor(matrix_32x32, cv2.COLOR_BGR2GRAY)
    face32x32 = cv2.resize(gray, (32,32)) / 255.0
    print(face32x32.shape)

    output_file = f"/run/media/gwolfovitch/Data/LAB-I/imagens_tratadas/imagens_teste_verilog/saidas/pixels_{i+1}_32x32_hex.txt"
    print(face32x32)

    with open(output_file, "w", encoding="utf-8") as f:
        for row in range(face32x32.shape[0]):
            for col in range(face32x32.shape[1]):
                value = face32x32[row, col]
                hex_value, _ = float_to_q1_7(value)
                f.write(f"{hex_value:02X}\n")


(32, 32)
[[0.67058824 0.69803922 0.71764706 ... 0.74117647 0.60392157 0.65098039]
 [0.67058824 0.71764706 0.79215686 ... 0.74509804 0.68235294 0.64313725]
 [0.67058824 0.70980392 0.68627451 ... 0.7254902  0.68235294 0.61568627]
 ...
 [0.6627451  0.67058824 0.65098039 ... 0.64705882 0.65882353 0.6627451 ]
 [0.6627451  0.67843137 0.6745098  ... 0.6627451  0.65882353 0.65490196]
 [0.67058824 0.6627451  0.65882353 ... 0.65098039 0.65882353 0.65882353]]
(32, 32)
[[0.73333333 0.49803922 0.03529412 ... 0.27058824 0.25098039 0.35294118]
 [0.69411765 0.15294118 0.05098039 ... 0.28235294 0.2627451  0.22352941]
 [0.66666667 0.10588235 0.12156863 ... 0.16078431 0.34117647 0.25490196]
 ...
 [0.12156863 0.03137255 0.05098039 ... 0.01960784 0.01568627 0.00784314]
 [0.20784314 0.09411765 0.05490196 ... 0.         0.03529412 0.05098039]
 [0.1254902  0.15294118 0.06666667 ... 0.02745098 0.00784314 0.10588235]]
(32, 32)
[[0.93333333 0.96862745 0.9372549  ... 0.88627451 0.92941176 0.93333333]
 [0.94509804